# Phase 1 — Face Recognition: Fine-tuning + Image Testing

Trains/fine-tunes the face embedding model used by `models/face/inference.py`, evaluates it (Experiment 1: accuracy / FAR / FRR / EER / ROC-AUC), and runs a dedicated **image-based testing** section with genuine/impostor pairs and a gallery-matching demo, per the project's Phase 1 requirements.

**Model:** `facenet-pytorch` InceptionResnetV1 pretrained on VGGFace2, fine-tuned with an ArcFace head (`models/common/arcface.py`).

**Dataset:** [LFW](http://vis-www.cs.umass.edu/lfw/) (Labeled Faces in the Wild), deepfunneled, downloaded directly via `sklearn.datasets.fetch_lfw_people` — no login required. Academic/research-use license; see `docs/DATASETS.md`.

**No Drive mount needed.** All artifacts are saved under `/content/repo/` and the trained checkpoint is downloaded at the end of this notebook.

## 1. Setup

In [ ]:
# --- Environment setup (no Google Drive mount required) ---
# Installs only what's missing on top of Colab's preinstalled torch/torchvision.
!pip install -q --no-deps facenet-pytorch
!pip install -q h5py

import sys, os
REPO_URL = "https://github.com/Malik8122/Cancelable-Multimodal-Biometric-Authentication-for-Critical-Infrastructure.git"
REPO_DIR = "/content/repo"
# Phase 1 code currently lives on this branch (not yet merged to main/master) -
# update to the default branch once the phase-1 PR is merged.
REPO_BRANCH = "phase-1-foundation"

if not os.path.isdir(REPO_DIR):
    !git clone -q -b $REPO_BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull -q

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Some hosted GPU sessions intermittently hand out a driver/torch-build
# combination where CUDA reports available but no kernel image exists for
# the actual device ("CUDA error: no kernel image is available for
# execution on the device") - smoke-test with a real op now and fall back
# to CPU rather than crashing deep into training on a broken GPU.
if DEVICE == "cuda":
    try:
        (torch.zeros(1, device=DEVICE) + 1).cpu()
    except Exception as e:  # torch.AcceleratorError, RuntimeError, etc.
        print(f"CUDA smoke test failed ({e}); falling back to CPU.")
        DEVICE = "cpu"
print("Using device:", DEVICE)

## 2. Download the dataset
`min_faces_per_person=20` keeps only identities with enough images for both training and genuine-pair evaluation.

In [ ]:
from sklearn.datasets import fetch_lfw_people
import numpy as np

lfw = fetch_lfw_people(min_faces_per_person=20, resize=1.0, color=True, funneled=True)
images = (lfw.images * 255).astype(np.uint8)  # (N, H, W, 3) RGB, already in [0,1]
labels = lfw.target
identity_names = lfw.target_names
print(f"{len(images)} images across {len(identity_names)} identities")

## 3. Preprocess (detect + align) every image
Reuses `preprocessing/face.py` so training sees exactly the same preprocessing the production pipeline will use at inference time.

In [ ]:
from preprocessing.face import FacePreprocessor

preprocessor = FacePreprocessor(device=DEVICE)
aligned_images, aligned_labels = [], []
skipped = 0
for img, label in zip(images, labels):
    try:
        aligned_images.append(preprocessor.preprocess(img))
        aligned_labels.append(label)
    except ValueError:
        skipped += 1  # no face detected in this crop; drop it rather than fail the run

aligned_images = np.stack(aligned_images)
aligned_labels = np.array(aligned_labels)
print(f"Aligned {len(aligned_images)} images, skipped {skipped} with no detected face")

## 4. Train / validation / test split
Split **per identity** so every split has genuine pairs (multiple images of the same person) available for verification evaluation.

In [ ]:
from collections import defaultdict

rng = np.random.default_rng(42)
by_identity = defaultdict(list)
for idx, label in enumerate(aligned_labels):
    by_identity[label].append(idx)

train_idx, val_idx, test_idx = [], [], []
for label, idxs in by_identity.items():
    idxs = np.array(idxs)
    rng.shuffle(idxs)
    n = len(idxs)
    n_train = max(1, int(n * 0.7))
    n_val = max(1, int(n * 0.15))
    train_idx.extend(idxs[:n_train])
    val_idx.extend(idxs[n_train:n_train + n_val])
    test_idx.extend(idxs[n_train + n_val:] if n - n_train - n_val > 0 else idxs[-1:])

print(f"train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

## 5. Model: InceptionResnetV1 backbone + ArcFace head

In [ ]:
import torch.nn as nn
import torch.optim as optim
from facenet_pytorch import InceptionResnetV1
from models.common.arcface import ArcMarginProduct
from models.face.inference import FACE_EMBEDDING_DIM

num_classes = len(identity_names)
backbone = InceptionResnetV1(pretrained='vggface2', classify=False).to(DEVICE)
arc_head = ArcMarginProduct(FACE_EMBEDDING_DIM, num_classes).to(DEVICE)

# Freeze everything except the last inception block + final linear layer:
# the pretrained VGGFace2 features are already strong; we only need to adapt
# the last few layers to this dataset, which is faster and less prone to
# overfitting on a few thousand images.
for name, param in backbone.named_parameters():
    param.requires_grad = any(name.startswith(p) for p in ['block8', 'last_linear', 'last_bn'])

trainable = [p for p in backbone.parameters() if p.requires_grad] + list(arc_head.parameters())
optimizer = optim.Adam(trainable, lr=1e-4)
criterion = nn.CrossEntropyLoss()

## 6. Training loop

In [ ]:
from torch.utils.data import DataLoader, Dataset

class FaceDataset(Dataset):
    def __init__(self, images, labels):
        self.images, self.labels = images, labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        tensor = torch.from_numpy(self.images[i]).permute(2, 0, 1).float()
        tensor = (tensor - 127.5) / 128.0
        return tensor, int(self.labels[i])

train_loader = DataLoader(FaceDataset(aligned_images[train_idx], aligned_labels[train_idx]), batch_size=32, shuffle=True)
val_loader = DataLoader(FaceDataset(aligned_images[val_idx], aligned_labels[val_idx]), batch_size=32)

NUM_EPOCHS = 10
for epoch in range(NUM_EPOCHS):
    backbone.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        embeddings = backbone(x)
        logits = arc_head(embeddings, y)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        train_correct += (logits.argmax(1) == y).sum().item()
        train_total += x.size(0)

    backbone.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            embeddings = backbone(x)
            logits = arc_head(embeddings, y)
            val_correct += (logits.argmax(1) == y).sum().item()
            val_total += x.size(0)

    print(f"epoch {epoch+1}/{NUM_EPOCHS} | train_loss={train_loss/train_total:.4f} "
          f"train_acc={train_correct/train_total:.3f} val_acc={val_correct/max(val_total,1):.3f}")

## 7. Save the checkpoint
Saves **only the backbone's** `state_dict` — this matches the architecture `models/face/inference.py` builds at inference time, so it loads back in directly with no extra glue code.

In [ ]:
from models.common.checkpoint_io import save_state_dict_as_h5

CHECKPOINT_PATH = f"{REPO_DIR}/models/face/saved/face_embedder.pt"
H5_PATH = f"{REPO_DIR}/models/face/saved/face_embedder.h5"
backbone.eval()
torch.save(backbone.state_dict(), CHECKPOINT_PATH)  # canonical, loaded by models/face/inference.py
save_state_dict_as_h5(backbone.state_dict(), H5_PATH)  # interoperability export
print("Saved checkpoint to", CHECKPOINT_PATH, "and", H5_PATH)

## 8. Experiment 1 — recognition performance on the held-out test set
Uses the shared `evaluation/` module so results are computed identically to how Phase 3's fusion evaluation will use them.

In [ ]:
from models.face.inference import FaceEmbedder
from evaluation.experiments import run_modality_experiment
from evaluation.roc import plot_roc

fine_tuned_embedder = FaceEmbedder(checkpoint_path=CHECKPOINT_PATH, device=DEVICE)
assert not fine_tuned_embedder.mock_mode, "Checkpoint failed to load — check the path above."

test_embeddings = [fine_tuned_embedder.extract_embedding(aligned_images[i]) for i in test_idx]
test_labels = [identity_names[aligned_labels[i]] for i in test_idx]

report = run_modality_experiment(test_embeddings, test_labels, modality_name="face")
print(f"Face  |  Accuracy@EER-threshold: {report['accuracy_at_eer_threshold']:.3f}  EER: {report['eer']:.3f}  AUC: {report['auc']:.3f}")
plot_roc({"Face (fine-tuned)": report["roc"]}, save_path=f"{REPO_DIR}/evaluation/results/face_roc.png")

## 9. Test on images
Genuine pair, impostor pair, and a small gallery-matching demo — concrete visual proof the pipeline works, not just a summary metric.

In [ ]:
import matplotlib.pyplot as plt
from evaluation.metrics import cosine_similarity

test_idx_arr = np.array(test_idx)
test_label_names = np.array([identity_names[aligned_labels[i]] for i in test_idx])

def pick_pair(same_identity: bool):
    for _ in range(200):
        i, j = rng.choice(len(test_idx_arr), size=2, replace=False)
        if (test_label_names[i] == test_label_names[j]) == same_identity:
            return test_idx_arr[i], test_idx_arr[j]
    raise RuntimeError("Could not find a suitable pair in 200 tries")

genuine_a, genuine_b = pick_pair(same_identity=True)
impostor_a, impostor_b = pick_pair(same_identity=False)
threshold = report["eer_threshold"]

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, (a, b, expected) in enumerate([(genuine_a, genuine_b, "GENUINE"), (impostor_a, impostor_b, "IMPOSTOR")]):
    emb_a = fine_tuned_embedder.extract_embedding(aligned_images[a])
    emb_b = fine_tuned_embedder.extract_embedding(aligned_images[b])
    score = cosine_similarity(emb_a, emb_b)
    verdict = "MATCH" if score >= threshold else "NO MATCH"
    for col, idx in enumerate([a, b]):
        axes[row, col].imshow(aligned_images[idx])
        axes[row, col].set_title(identity_names[aligned_labels[idx]], fontsize=9)
        axes[row, col].axis("off")
    fig.text(0.5, 1 - row * 0.5 - 0.03, f"{expected} pair — similarity={score:.3f} → {verdict} (threshold={threshold:.3f})",
             ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{REPO_DIR}/evaluation/results/face_pair_test.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Gallery-matching demo: enroll one image per identity, then match 5 random query images against the gallery.
gallery = {}
for i in test_idx_arr:
    name = identity_names[aligned_labels[i]]
    if name not in gallery:
        gallery[name] = fine_tuned_embedder.extract_embedding(aligned_images[i])
gallery_names = list(gallery.keys())
gallery_matrix = np.stack(list(gallery.values()))

query_indices = rng.choice(test_idx_arr, size=min(5, len(test_idx_arr)), replace=False)
fig, axes = plt.subplots(1, len(query_indices), figsize=(3 * len(query_indices), 3.5))
if len(query_indices) == 1:
    axes = [axes]
for ax, idx in zip(axes, query_indices):
    query_emb = fine_tuned_embedder.extract_embedding(aligned_images[idx])
    sims = gallery_matrix @ query_emb  # embeddings are L2-normalized, so this is cosine similarity
    best = int(np.argmax(sims))
    predicted, true = gallery_names[best], identity_names[aligned_labels[idx]]
    ax.imshow(aligned_images[idx])
    correct = "✓" if predicted == true else "✗"
    ax.set_title(f"true: {true}\npred: {predicted} {correct}\nsim={sims[best]:.3f}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{REPO_DIR}/evaluation/results/face_gallery_test.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Get the checkpoint back to your machine
**Option A (recommended):** download the file below, then on your own machine:
```bash
cp face_embedder.pt models/face/saved/
git add models/face/saved/face_embedder.pt
git commit -m "Add fine-tuned face embedding checkpoint"
git push
```
(`git lfs` is already tracking `*.pt` via `.gitattributes` — no extra setup needed.)

**Option B:** push directly from Colab by setting a GitHub token as a Colab secret (`Secrets` panel, key `GITHUB_TOKEN`) — never hard-code a token in the notebook itself.

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)
files.download(H5_PATH)